In [9]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

In [25]:
df = pd.read_csv(r"D:\MFT\DL\battery_synthetic_dataset.csv")

In [27]:
df.head

<bound method NDFrame.head of      avg_voltage  avg_temperature  battery_life_months  cycle_count  label
0          3.327            39.19                 61.5         1127      0
1          3.419            34.96                 40.8          861      0
2          3.456            44.52                 59.3         1131      0
3          3.292            39.37                 49.5         1096      0
4          3.196            35.65                 48.1         1045      0
..           ...              ...                  ...          ...    ...
995        3.863            30.81                 21.7          435      1
996        3.338            40.38                 49.9          952      0
997        3.768            31.60                 39.0          723      1
998        3.922            26.51                  1.0           32      1
999        3.603            34.32                 33.0          670      1

[1000 rows x 5 columns]>

In [29]:
data = df.copy()
x = data.drop("label", axis=1)
y = data["label"]

In [33]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [35]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [37]:
input_size = x_train.shape[1]
hidden1_size = 5
hidden2_size = 5
output_size = 1

In [39]:
def sigmoid(x):
    return 1/(1+np.exp(-x))

def sigmoid_derivative(x):
    return x*(1-x)

In [43]:
rng = np.random.default_rng(42)

def he_init(fan_in, fan_out):
    return rng.standard_normal((fan_in, fan_out)) * np.sqrt(2/fan_in)

In [47]:
weight_input_hidden1 = he_init(input_size, hidden1_size)
bias_input_hidden1 = np.zeros((1, hidden1_size))

weight_hidden1_hidden2 = he_init(hidden1_size, hidden2_size)
bias_hidden1_hidden2 = np.zeros((1, hidden2_size))

weight_hidden2_output = he_init(hidden2_size, output_size)
bias_hidden2_output = np.zeros((1, output_size))

In [49]:
epochs = 10000
learning_rate = 1e-2
n = x_train.shape[0]

In [ ]:
##### tested 4, 5, 6, 7, 10 neurons and chose 5,5 as it gave the best results

In [55]:
print("Training 2-Hidden Layer Sigmoid Model...")
print("="*50)

for epoch in range(epochs):
    # Forward pass
    hidden1_input = np.dot(x_train, weight_input_hidden1) + bias_input_hidden1
    hidden1_output = sigmoid(hidden1_input)
    
    hidden2_input = np.dot(hidden1_output, weight_hidden1_hidden2) + bias_hidden1_hidden2
    hidden2_output = sigmoid(hidden2_input)
    
    output_input = np.dot(hidden2_output, weight_hidden2_output) + bias_hidden2_output
    output = sigmoid(output_input)
    
    error = y_train.values.reshape(-1, 1) - output
    delta_output = error * sigmoid_derivative(output)
    delta_hidden2 = delta_output.dot(weight_hidden2_output.T) * sigmoid_derivative(hidden2_output)
    
    delta_hidden1 = delta_hidden2.dot(weight_hidden1_hidden2.T) * sigmoid_derivative(hidden1_output)
    gradient_hidden2_output = hidden2_output.T.dot(delta_output) / n
    gradient_hidden1_hidden2 = hidden1_output.T.dot(delta_hidden2) / n
    gradient_input_hidden1 = x_train.T.dot(delta_hidden1) / n
    
    weight_hidden2_output += learning_rate * gradient_hidden2_output
    bias_hidden2_output += learning_rate * np.mean(delta_output, axis=0, keepdims=True)
    
    weight_hidden1_hidden2 += learning_rate * gradient_hidden1_hidden2
    bias_hidden1_hidden2 += learning_rate * np.mean(delta_hidden2, axis=0, keepdims=True)
    
    weight_input_hidden1 += learning_rate * gradient_input_hidden1
    bias_input_hidden1 += learning_rate * np.mean(delta_hidden1, axis=0, keepdims=True)
    
    if (epoch+1) % 1000 == 0 or epoch == 0:
        loss = np.mean(np.square(error))
        print(f"Epoch {epoch+1}/{epochs} - Loss: {loss}")


print("="*50)
print("training complete")
print("="*50)

Training 2-Hidden Layer Sigmoid Model...
Epoch 1/10000 - Loss: 0.08446582996255307
Epoch 1000/10000 - Loss: 0.08279555167325935
Epoch 2000/10000 - Loss: 0.08133336239668192
Epoch 3000/10000 - Loss: 0.08004745759542403
Epoch 4000/10000 - Loss: 0.0789105543848967
Epoch 5000/10000 - Loss: 0.07790037409389365
Epoch 6000/10000 - Loss: 0.07699861720803927
Epoch 7000/10000 - Loss: 0.07619016184762174
Epoch 8000/10000 - Loss: 0.07546243661815838
Epoch 9000/10000 - Loss: 0.07480492855753113
Epoch 10000/10000 - Loss: 0.07420879553004585
training complete


In [57]:
hidden1_input = np.dot(x_test, weight_input_hidden1) + bias_input_hidden1
hidden1_output = sigmoid(hidden1_input)

hidden2_input = np.dot(hidden1_output, weight_hidden1_hidden2) + bias_hidden1_hidden2
hidden2_output = sigmoid(hidden2_input)

output_input = np.dot(hidden2_output, weight_hidden2_output) + bias_hidden2_output
output = sigmoid(output_input)

In [59]:
print("\n" + "="*50)
print("Classification Report (Threshold = 0.5)")
print("="*50)

predicted_labels = (output > 0.5).astype(int)
print(classification_report(y_test, predicted_labels))
print(f"Accuracy: {accuracy_score(y_test, predicted_labels):.4f}")


Classification Report (Threshold = 0.5)
              precision    recall  f1-score   support

           0       0.83      0.93      0.88        73
           1       0.96      0.89      0.92       127

    accuracy                           0.91       200
   macro avg       0.89      0.91      0.90       200
weighted avg       0.91      0.91      0.91       200

Accuracy: 0.9050


In [61]:
print("\n" + "="*50)
print("Threshold Analysis for Different Thresholds")
print("="*50)

thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
results = []
best_acc = 0
best_f1 = 0
best_thresh_acc = 0.5
best_thresh_f1 = 0.5

print(f"{'Threshold':<10} {'Accuracy':<10} {'Precision':<10} {'Recall':<10} {'F1 Score':<10}")
print("-"*60)


for thresh in thresholds:
    preds = (output > thresh).astype(int)
    acc = accuracy_score(y_test, preds)
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    results.append({
        'threshold': thresh,
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp
    })
    
    print(f"{thresh:<10.1f} {acc:<10.4f} {precision:<10.4f} {recall:<10.4f} {f1:<10.4f}")
    
    if acc > best_acc:
        best_acc = acc
        best_thresh_acc = thresh
    if f1 > best_f1:
        best_f1 = f1
        best_thresh_f1 = thresh

print("-"*60)


Threshold Analysis for Different Thresholds
Threshold  Accuracy   Precision  Recall     F1 Score  
------------------------------------------------------------
0.1        0.6350     0.6350     1.0000     0.7768    
0.2        0.8450     0.8038     1.0000     0.8912    
0.3        0.9050     0.8913     0.9685     0.9283    
0.4        0.9100     0.9291     0.9291     0.9291    
0.5        0.9050     0.9576     0.8898     0.9224    
0.6        0.9050     0.9909     0.8583     0.9198    
0.7        0.8650     0.9902     0.7953     0.8821    
0.8        0.8200     1.0000     0.7165     0.8349    
0.9        0.7450     1.0000     0.5984     0.7488    
------------------------------------------------------------


In [63]:
print("\n" + "="*50)
print("Summary of Best Thresholds")
print("="*50)
print(f"Best threshold by Accuracy: {best_thresh_acc:.1f} (Accuracy = {best_acc:.4f})")
print(f"Best threshold by F1 Score: {best_thresh_f1:.1f} (F1 = {best_f1:.4f})")


Summary of Best Thresholds
Best threshold by Accuracy: 0.4 (Accuracy = 0.9100)
Best threshold by F1 Score: 0.4 (F1 = 0.9291)


In [67]:
print("\n" + "="*50)
print(f"Classification Report (Best Threshold = {best_thresh_f1:.1f})")
print("="*50)
best_preds = (output > best_thresh_f1).astype(int)
print(classification_report(y_test, best_preds))
print(f"Accuracy: {accuracy_score(y_test, best_preds):.4f}")

print("\nConfusion Matrix for Best Thresh:")
cm = confusion_matrix(y_test, best_preds)
print(f"[[TN: {cm[0][0]}, FP: {cm[0][1]}]")
print(f" [FN: {cm[1][0]}, TP: {cm[1][1]}]")


Classification Report (Best Threshold = 0.4)
              precision    recall  f1-score   support

           0       0.88      0.88      0.88        73
           1       0.93      0.93      0.93       127

    accuracy                           0.91       200
   macro avg       0.90      0.90      0.90       200
weighted avg       0.91      0.91      0.91       200

Accuracy: 0.9100

Confusion Matrix for Best Thresh:
[[TN: 64, FP: 9]
 [FN: 9, TP: 118]


In [69]:
print("\n" + "="*50)
print("Detailed Results for All Thresholds")
print("="*50)
print(f"{'Thresh':<8} {'Acc':<8} {'Prec':<8} {'Recall':<8} {'F1':<8} {'TN':<6} {'FP':<6} {'FN':<6} {'TP':<6}")
print("-"*70)
for r in results:
    print(f"{r['threshold']:<8.1f} {r['accuracy']:<8.4f} {r['precision']:<8.4f} {r['recall']:<8.4f} {r['f1_score']:<8.4f} {r['tn']:<6} {r['fp']:<6} {r['fn']:<6} {r['tp']:<6}")


Detailed Results for All Thresholds
Thresh   Acc      Prec     Recall   F1       TN     FP     FN     TP    
----------------------------------------------------------------------
0.1      0.6350   0.6350   1.0000   0.7768   0      73     0      127   
0.2      0.8450   0.8038   1.0000   0.8912   42     31     0      127   
0.3      0.9050   0.8913   0.9685   0.9283   58     15     4      123   
0.4      0.9100   0.9291   0.9291   0.9291   64     9      9      118   
0.5      0.9050   0.9576   0.8898   0.9224   68     5      14     113   
0.6      0.9050   0.9909   0.8583   0.9198   72     1      18     109   
0.7      0.8650   0.9902   0.7953   0.8821   72     1      26     101   
0.8      0.8200   1.0000   0.7165   0.8349   73     0      36     91    
0.9      0.7450   1.0000   0.5984   0.7488   73     0      51     76    
